# BDS Việt Nam — Báo cáo Tổng hợp Thị trường Bất động sản

> **Generated by AI Lab DataHub** · `bds-bao-cao-tong-hop` v1.0.0 · 2026-06-12T09:44:18.255664+00:00

Báo cáo tổng hợp thị trường BĐS Việt Nam 2024Q1–2026Q1 từ 24 báo cáo tháng/quý: vĩ mô 9 chỉ số, cung-cầu-giá theo quý (nguồn Bộ Xây dựng), 7 figures. Data tải tự động từ Google Drive khi chạy trên Colab.

**License**: CC-BY-4.0  
**Citation**: UEL FinTech Lab (2026). BDS Việt Nam — Báo cáo Tổng hợp Thị trường Bất động sản. DataHub n2nai.  
**Tags**: bat-dong-san, real-estate, vietnam, macro, quarterly  
**Runtime**: 🟢 Colab Free tier OK


## Quick start

1. **Runtime → Run all** — notebook đã có sẵn API Key (auto-injected khi publish).
2. Nếu cần rotate key: vào [DataHub Workspace](https://datahub.n2nai.io/workspace) → Project `uel-fintech` → tab **API Gateway**.

_Không cần tạo Colab Secret — key đã embed trong Cell 1._


## Cell 1 — Setup + auth check


In [ ]:
!pip install -q requests pandas pyarrow
import os, io, requests, pandas as pd

BASE_URL = "https://datahub.n2nai.io"
PROJECT_ID = 29
PROJECT_NAME = "uel-fintech"
PRODUCT_SLUG = "bds-bao-cao-tong-hop"

# ── DataHub API Key (auto-injected by publish — rotate qua UI Workspace → API Keys) ──
DATAHUB_API_KEY = "prj_uel-fintech_vgiQt9ChTjAXXxdi_sU6L6xCsC-jDeFpOMPi8Y1oRl4"
HEADERS = {'X-API-Key': DATAHUB_API_KEY}

# Verify key OK + show scope
r = requests.get(f'{BASE_URL}/api/v1/projects/{PROJECT_NAME}/api-keys/whoami', headers=HEADERS)
print('Auth status:', r.status_code)
print(r.json() if r.ok else r.text)

## Source notebook content

Inlined from `BDS/00_bao_cao_tong_hop_BDS.ipynb` — version `1.0.0`.

Bao gồm 22 cell gốc từ notebook owner đã publish.


# 📊 Báo cáo tổng hợp — Dữ liệu Bất động sản & Vĩ mô Việt Nam
### Khai thác trọn vẹn kho 24 báo cáo REFI/AFA Capital (T5/2024 → T5/2026)

**Project**: uel-fintech · **Folder**: `work/BDS` · **Ngày tổng hợp**: 2026-06-12

---
## Mục lục
1. [Tổng quan & Kiến trúc Data Lake 6 lớp](#1)
2. [Quy trình xử lý dữ liệu (7 bước)](#2)
3. [Inventory — 11 bộ dữ liệu đã sinh](#3)
4. [Vĩ mô hàng tháng (CPI, PMI, XNK...)](#4)
5. [Nguồn cung BĐS theo quý (toàn quốc)](#5)
6. [Giao dịch BĐS — quý & năm](#6)
7. [Tồn kho BĐS 12 quý](#7)
8. [Nguồn cung theo thành phố (HN / ĐN / TP.HCM)](#8)
9. [Mặt bằng giá theo vùng](#9)
10. [Key findings & Đề xuất mở rộng](#10)


<a id="1"></a>
## 1. Tổng quan & Kiến trúc Data Lake 6 lớp

Nguồn gốc: **24 báo cáo PDF** (báo cáo tháng + 2 báo cáo quý + báo cáo Chiến lược) của REFI/AFA Capital,
trích dẫn số liệu từ **Bộ Xây dựng, GSO, HSC, KBSV, DXS, Widata**...

```
L1 SOURCES        PDF báo cáo tháng/quý (REFI, AFA Capital)
   │
L2 RAW STAGING    MinIO: PDF gốc + PNG trang chart (vision_pages/)
   │
L3 BRONZE         Full text 24 báo cáo (text/*.txt) — pdftotext
   │
L4 SILVER         11 CSV chuẩn hóa (macro + 9 bộ BĐS + catalog)
   │
L5 GOLD           Panel tổng hợp tháng/quý (notebook này minh họa)
   │
L6 SERVING        Notebook / Slide / Dashboard / RAG Q&A
```

**Thu thập hàng tháng**: Airflow DAG `bds_monthly_ingest` — có PDF tháng mới chỉ cần
drop vào folder → chạy lại `extract_datasets.py` (idempotent, báo cáo mới ghi đè kỳ trùng).


<a id="2"></a>
## 2. Quy trình xử lý dữ liệu (7 bước đã thực hiện)

```
① Thu thập & chuẩn hóa tên file (24 PDF)
        ▼
② pdftotext → Bronze full-text (text/*.txt)
        ▼
③ Regex parse bảng vĩ mô 12 tháng / báo cáo
        ▼
④ Hợp nhất + dedup (báo cáo mới ghi đè kỳ trùng)
        ▼
⑤ Quality gate — cắt dữ liệu trước 2024-08 (độ tin cậy thấp)
        ▼
⑥ Xuất Silver CSV (macro_monthly, reports_catalog)
        ▼
⑦ LLM VISION EXTRACTION — số liệu nằm trong CHART ẢNH:
   • Scan tiêu đề trang toàn kho → khoanh trang chart (≈90 trang)
   • Render PNG 110→200dpi (pdftoppm) các trang trọng tâm
   • Claude vision đọc data label → CSV + cột confidence + source_page (lineage)
   • Quality loop: re-run hi-DPI trang label mờ → 100% high confidence
   • Cross-validation: HN/HCM xuất hiện ở 2 báo cáo độc lập → 28/28 điểm khớp
```

**Phát hiện then chốt ở bước ⑦**: Bộ Xây dựng chỉ công bố cung/giao dịch theo **QUÝ**
(báo cáo ra hàng tháng ≠ số liệu theo tháng) → chiến lược chuyển sang quét báo cáo
Chiến lược/quý chứa chuỗi lịch sử dài nhất, tiết kiệm ~97% chi phí vision.


<a id="3"></a>
## 3. Inventory — 11 bộ dữ liệu đã sinh (Silver layer)

In [ ]:
# ============================================================
# 🔧 COLAB SETUP — tự tải datasets từ Google Drive (RealEstate-Refi)
# Chạy trên DataHub workspace (đã có folder datasets/) → cell tự bỏ qua.
# ============================================================
import os
if not os.path.isdir('datasets'):
    import urllib.request
    os.makedirs('datasets', exist_ok=True)
    FILES = {
        "macro_monthly.csv": "1Cek7_qt9vHAkaUCIritFhBUXbov6NWH8",
        "re_city_apartment.csv": "10rtSPV0vVpP6N2dDuUxvgXCiK1RqXH7F",
        "re_inventory_quarterly.csv": "1kZMFtCr5Dx1xijEfsTAmk48YgStHPE4W",
        "re_price_q1_2026.csv": "1zXbRxBNMq7pi55ovs_773ZYBQFO3L8x-",
        "re_price_range_2025.csv": "1PgaeTQemaf1mzvxa_R2GZO8PG_D6CffE",
        "re_supply_city_quarterly.csv": "11b95CVjF6Qcx6jy4H1ynHDkYjGmucKLB",
        "re_supply_hcm_region_2025q3.csv": "1kCNiUH_8nFtEwpXNIdQ22RB22tl5jzC1",
        "re_supply_quarterly.csv": "1kLR10gH5LUdnueWWa8IOlkkP1WAwj3SU",
        "re_transactions_annual.csv": "1Pkhj9b15q0dzqczsyDKjRsL3XzUpp7Po",
        "re_transactions_quarterly.csv": "19FK-iopzn2qcaAE4hLock4Wd2-15M23h",
        "reports_catalog.csv": "1RCyyny4nopWzApTZR94UfEQsFkXpl1gH"
}
    for name, fid in FILES.items():
        urllib.request.urlretrieve(
            f'https://drive.google.com/uc?export=download&id={fid}',
            os.path.join('datasets', name))
    print(f'✅ Đã tải {len(FILES)} datasets từ Google Drive folder RealEstate-Refi')
else:
    print('✅ Folder datasets/ đã có sẵn (DataHub workspace) — bỏ qua download')


In [ ]:
import pandas as pd, matplotlib, matplotlib.pyplot as plt, os, glob
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 110, 'axes.unicode_minus': False})
DATA = 'datasets'

FILES = {
 'macro_monthly.csv':            'Vĩ mô 9 chỉ số × tháng (2024-08 → 2026-04)',
 'reports_catalog.csv':          'Metadata 24 báo cáo (tháng, số trang, nguồn)',
 're_supply_quarterly.csv':      'Nguồn cung BĐS toàn quốc theo quý',
 're_transactions_quarterly.csv':'Giao dịch BĐS theo quý 2025-2026',
 're_transactions_annual.csv':   'Giao dịch cả năm 2022-2024',
 're_inventory_quarterly.csv':   'Tồn kho 12 quý 2023Q1→2025Q4 × 3 phân khúc',
 're_supply_city_quarterly.csv': 'Cung mới HN/ĐN/HCM × 2 loại hình (2022→2025Q3)',
 're_supply_hcm_region_2025q3.csv':'Cơ cấu TP.HCM mới sau sáp nhập Q3/2025',
 're_city_apartment.csv':        'Căn hộ theo thành phố (cung/giá)',
 're_price_q1_2026.csv':         'Giá theo vùng Q1/2026 (min-max + QoQ)',
 're_price_range_2025.csv':      'Khung giá 2025 theo vùng + YoY',
}
rows = []
dfs = {}
for f, desc in FILES.items():
    p = os.path.join(DATA, f)
    df = pd.read_csv(p)
    dfs[f] = df
    conf = ''
    if 'confidence' in df.columns:
        conf = f"{(df['confidence']=='high').mean()*100:.0f}% high"
    rows.append({'file': f, 'rows': len(df), 'cols': len(df.columns), 'confidence': conf, 'mô tả': desc})
inv = pd.DataFrame(rows)
print(f"✅ Loaded {len(dfs)}/11 datasets — tổng {inv['rows'].sum()} dòng")
inv

<a id="4"></a>
## 4. Vĩ mô hàng tháng

Panel 9 chỉ số: CPI, bán lẻ, XK/NK, cán cân thương mại, IIP, PMI, lãi suất ON, USD/VND.
**Vùng tin cậy: từ 2024-08** (dữ liệu trước đó trích từ báo cáo Chiến lược, đã được quality gate loại khi phân tích).

In [ ]:
macro = dfs['macro_monthly.csv'].copy()
macro['month'] = pd.to_datetime(macro['month'])
m = macro[macro['month'] >= '2024-08'].set_index('month')
print(f"Vùng tin cậy: {m.index.min():%Y-%m} → {m.index.max():%Y-%m} ({len(m)} tháng)")
print("NaN count/cột:"); print(m.isna().sum())
m.tail(6)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(m.index, m['CPI (YoY)'], 'o-', color='#d62728', label='CPI YoY (%)')
ax1.set_ylabel('CPI YoY (%)', color='#d62728'); ax1.tick_params(axis='y', labelcolor='#d62728')
ax2 = ax1.twinx()
ax2.plot(m.index, m['PMI'], 's--', color='#1f77b4', label='PMI')
ax2.axhline(50, color='gray', lw=0.8, ls=':')
ax2.set_ylabel('PMI (điểm)', color='#1f77b4'); ax2.tick_params(axis='y', labelcolor='#1f77b4')
ax1.set_title(f'CPI YoY vs PMI — {len(m)} tháng (2024-08 → 2026-04)', fontsize=13)
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left', bbox_to_anchor=(0, -0.08), ncol=3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(m.index, m['Xuất khẩu (YoY)'], width=20, label='Xuất khẩu YoY (%)', color='#2ca02c', alpha=.8)
ax.bar(m.index, -m['Nhập khẩu (YoY)'].abs()*0, width=0)  # giữ trục
ax.plot(m.index, m['Nhập khẩu (YoY)'], 'o-', color='#ff7f0e', label='Nhập khẩu YoY (%)')
ax.set_title('Xuất khẩu vs Nhập khẩu (YoY %)', fontsize=13)
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.08), ncol=2)
plt.tight_layout(); plt.show()

<a id="5"></a>
## 5. Nguồn cung BĐS theo quý (toàn quốc — Bộ Xây dựng)

Số liệu **số hóa từ chart ảnh** bằng LLM vision (bước ⑦), đã quality-loop hi-DPI → 100% high confidence.

In [ ]:
sup = dfs['re_supply_quarterly.csv'].copy()
print("Metrics:", sup['metric'].unique().tolist())
pv = sup.pivot_table(index='quarter', columns='metric', values='value', aggfunc='first')
pv

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if 'products_under_development' in pv.columns:
    s = pv['products_under_development'].dropna()
    axes[0].bar(s.index, s.values/1000, color='#1f77b4')
    axes[0].set_title('Sản phẩm đang triển khai (nghìn SP)', fontsize=12)
    for x, v in zip(s.index, s.values): axes[0].text(x, v/1000, f'{v/1000:,.0f}', ha='center', va='bottom', fontsize=9)
other = [c for c in pv.columns if c != 'products_under_development']
for ccol in other:
    s = pv[ccol].dropna()
    axes[1].plot(s.index, s.values/1000, 'o-', label=ccol)
axes[1].set_title('Các chuỗi cung khác (nghìn căn)', fontsize=12)
axes[1].legend(loc='upper left', bbox_to_anchor=(0, -0.12), ncol=2, fontsize=9)
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

<a id="6"></a>
## 6. Giao dịch BĐS — theo quý & cả năm

In [ ]:
trq = dfs['re_transactions_quarterly.csv'].copy()
tra = dfs['re_transactions_annual.csv'].copy()
pvq = trq.pivot_table(index='quarter', columns='segment', values='transactions', aggfunc='first')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pvq.plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('Giao dịch theo quý (2025→2026)', fontsize=12)
axes[0].legend(loc='upper left', bbox_to_anchor=(0, -0.18), ncol=2, fontsize=9)
axes[1].bar(tra['year'].astype(str), tra['chungcu_nha_rieng_le']/1000, label='CHCC + nhà riêng lẻ', color='#1f77b4')
axes[1].bar(tra['year'].astype(str), tra['dat_nen']/1000, bottom=tra['chungcu_nha_rieng_le']/1000, label='Đất nền', color='#ff7f0e')
axes[1].set_title('Giao dịch cả năm (nghìn GD) — 2022-2024', fontsize=12)
axes[1].legend(loc='upper left', bbox_to_anchor=(0, -0.12), ncol=2, fontsize=9)
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print("Tổng giao dịch năm:"); print(tra[['year','total']].to_string(index=False))

<a id="7"></a>
## 7. Tồn kho BĐS 12 quý (2023Q1 → 2025Q4) — series mới hoàn toàn

In [ ]:
invq = dfs['re_inventory_quarterly.csv'].copy()
pvi = invq.pivot_table(index='quarter', columns='segment', values='inventory_units', aggfunc='first')
fig, ax = plt.subplots(figsize=(12, 5.5))
for ccol in pvi.columns:
    ax.plot(pvi.index, pvi[ccol]/1000, 'o-', label=ccol)
ax.set_title('Tồn kho theo phân khúc (nghìn căn/lô/nền) — nguồn: Bộ Xây dựng', fontsize=13)
ax.set_ylabel('nghìn căn/lô/nền')
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.12), ncol=3, fontsize=10)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

<a id="8"></a>
## 8. Nguồn cung mới theo thành phố — HN / ĐN / TP.HCM

Cross-validation 2 báo cáo quý độc lập (08/2025 & 11/2025): **28/28 điểm overlap khớp 100%**.

In [ ]:
city = dfs['re_supply_city_quarterly.csv'].copy()
cq = city[city['granularity'] == 'quarter']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cty, g in cq.groupby('city'):
    axes[0].plot(g['period'], g['chung_cu']/1000, 'o-', label=cty)
    axes[1].plot(g['period'], g['bds_gan_lien_dat']/1000, 's--', label=cty)
axes[0].set_title('Cung mới CHUNG CƯ (nghìn căn)', fontsize=12)
axes[1].set_title('Cung mới BĐS GẮN LIỀN ĐẤT (nghìn căn)', fontsize=12)
for ax in axes:
    ax.tick_params(axis='x', rotation=60)
    ax.legend(loc='upper left', bbox_to_anchor=(0, -0.25), ncol=3, fontsize=9)
plt.tight_layout(); plt.show()
hcm = dfs['re_supply_hcm_region_2025q3.csv']
print('Cơ cấu TP.HCM mới sau sáp nhập (Q3/2025):')
print(hcm[['region','can_ho_chung_cu','bds_gan_lien_dat','note']].to_string(index=False))

<a id="9"></a>
## 9. Mặt bằng giá theo vùng (khung min–max, triệu đồng/m²)

In [ ]:
pr = dfs['re_price_q1_2026.csv'].copy()
pr25 = dfs['re_price_range_2025.csv'].copy()
fig, ax = plt.subplots(figsize=(12, 6))
labels, y = [], 0
colors = {'chung_cu': '#1f77b4', 'nha_pho': '#ff7f0e', 'dat_nen': '#2ca02c'}
for _, r in pr.iterrows():
    ax.barh(y, r['price_max'] - r['price_min'], left=r['price_min'],
            color=colors.get(r['segment'], '#7f7f7f'), alpha=.85)
    ax.text(r['price_max'] + 2, y, f"{r['price_min']:.0f}–{r['price_max']:.0f} ({r['trend_qoq']})", va='center', fontsize=9)
    labels.append(f"{r['region']} · {r['segment']}"); y += 1
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('triệu đồng/m²')
ax.set_title(f'Khung giá Q1/2026 theo vùng × phân khúc (n={len(pr)}) — kèm xu hướng QoQ', fontsize=13)
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color=v, label=k) for k, v in colors.items()],
          loc='upper left', bbox_to_anchor=(0, -0.08), ncol=3)
plt.tight_layout(); plt.show()
print('Khung giá 2025 (YoY):'); print(pr25[['region','segment','price_min_tr_m2','price_max_tr_m2','yoy_change']].head(10).to_string(index=False))

<a id="10"></a>
## 10. Key findings & Đề xuất mở rộng

### 🔑 Key findings
| # | Phát hiện | Nguồn |
|---|---|---|
| 1 | Sản phẩm BĐS đang triển khai đạt **564.388 SP** Q1/2026 (tăng từ ~400k đầu 2025) | `re_supply_quarterly` |
| 2 | Nhà ở đủ điều kiện mở bán tăng **~6 lần** trong 2 năm: 5.572 căn (2024Q1) → 34.315 căn (2026Q1) | `re_supply_quarterly` |
| 3 | Giao dịch đất nền luôn **gấp ~3 lần** CHCC+nhà riêng lẻ theo quý | `re_transactions_quarterly` |
| 4 | Giao dịch cả năm 2022 đạt đỉnh 785.637, giảm sâu 2023 còn 433.897 | `re_transactions_annual` |
| 5 | Bình Dương (cũ) đóng góp **~75%** nguồn cung căn hộ TP.HCM mới 9T/2025 | `re_supply_hcm_region` |
| 6 | Giá chung cư miền Bắc 45–170 tr/m², YoY 2025 tăng **20-30%** | `re_price_*` |
| 7 | Bộ Xây dựng chỉ công bố cung/giao dịch theo **QUÝ** — không tồn tại chuỗi tháng ở nguồn | Lesson bước ⑦ |

### 🧭 Data lineage & độ tin cậy
- Mọi dòng số liệu vision đều có `confidence` + `source_page` (+ `source_report`, `source_org`) → truy ngược về trang PDF gốc.
- 100% dòng đạt `confidence=high` sau quality loop hi-DPI + cross-validation 2 nguồn độc lập.

### 🚀 Đề xuất mở rộng
1. **Airflow DAG `bds_monthly_ingest`** chạy production — tự ingest PDF tháng mới → append Silver.
2. **Gold layer**: join panel vĩ mô × BĐS quý → phân tích tương quan (lãi suất ↔ giao dịch).
3. **RAG Q&A**: ingest `text/*.txt` (24 báo cáo) vào knowledge collection → hỏi đáp chính sách/sự kiện.
4. **Dashboard Superset**: 4 chart chính của notebook này làm dashboard theo dõi hàng quý.
5. **Publish Data Product**: đóng gói 11 CSV + notebook này ra catalog Colab.

---
*Notebook sinh tự động bởi DataHub Orchestrator — 2026-06-12. Tái chạy: `jupyter nbconvert --execute 00_bao_cao_tong_hop_BDS.ipynb`*


## Reproducibility lock

Notebook này pin tới Data Product version `1.0.0` — kết quả reproducible bất kể source data thay đổi sau này.


In [ ]:
print('DataHub Product: bds-bao-cao-tong-hop v1.0.0')
print('Generated: 2026-06-12T09:44:18.255664+00:00')
print(f'Pandas: {pd.__version__}')